# CFP_LagLlama_Forecasts

Generates 1-step-ahead VaR forecasts using Lag-Llama (7.5M) via Student-t distribution head with lag features. 1000 samples per day.

**Paper:** Pele, Lessmann, Hardle (2026)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path('../cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'

ASSETS = [
    'SP500','STOXX','GDAXI','CACT','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
CONTEXT = 512
N_SAMPLES = 1000

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

print(f'Assets: {len(ASSETS)}, Context: {CONTEXT}, Samples: {N_SAMPLES}')

MODELS = [('lagllama', 'time-series-foundation-models/Lag-Llama', 'Lag-Llama')]

In [ ]:
import torch
from huggingface_hub import hf_hub_download
import sys
# Clone lag-llama repo first: git clone https://github.com/time-series-foundation-models/lag-llama.git
sys.path.insert(0, '../lag-llama')
from lag_llama.gluon.estimator import LagLlamaEstimator
from gluonts.dataset.pandas import PandasDataset

ckpt = hf_hub_download(repo_id='time-series-foundation-models/Lag-Llama', filename='lag-llama.ckpt')
estimator = LagLlamaEstimator(
    ckpt_path=ckpt, prediction_length=1, context_length=CONTEXT, num_samples=N_SAMPLES,
    nonnegative_pred_samples=False,
)
predictor = estimator.create_lightning_module().to('cpu')

out_dir = DATA_DIR / 'lagllama'
out_dir.mkdir(exist_ok=True)

for asset in tqdm(ASSETS, desc='Lag-Llama'):
    ret = load_returns(asset)
    n = len(ret)
    vals = ret.values
    dates = ret.index
    records = []
    for t in range(CONTEXT, n):
        context_df = pd.DataFrame({'target': vals[t-CONTEXT:t]},
                                   index=pd.period_range(dates[t-CONTEXT], periods=CONTEXT, freq='D'))
        ds = PandasDataset(dict(context_df), target='target', freq='D')
        forecasts = list(predictor.predict(ds, num_samples=N_SAMPLES))
        samples = forecasts[0].samples.flatten()
        row = {'date': dates[t]}
        for alpha in ALPHAS:
            row[f'VaR_{alpha}'] = np.percentile(samples, alpha * 100)
        row['mean'] = samples.mean()
        row['std'] = samples.std()
        records.append(row)
    df_out = pd.DataFrame(records).set_index('date')
    df_out.to_parquet(out_dir / f'{asset}.parquet')
print(f'Lag-Llama: {len(ASSETS)} assets saved')

In [ ]:
# Verify outputs
from pathlib import Path
for model_slug, _, label in MODELS:
    out_dir = DATA_DIR / model_slug
    files = list(out_dir.glob('*.parquet'))
    print(f'{label}: {len(files)} parquet files')